In [6]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers, models, callbacks
import os
train_dir = r"D:\PROJECT\Agriculture\Master_Dataset\train"
val_dir = r"D:\PROJECT\Agriculture\Master_Dataset\val"

In [7]:
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=90,           
    width_shift_range=0.3,       
    height_shift_range=0.3,
    shear_range=0.3,            
    zoom_range=0.4,              
    horizontal_flip=True,        
    vertical_flip=True,          
    brightness_range=[0.5, 1.5],
    fill_mode='reflect'          
)

val_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(299, 299),
    batch_size=32,
    class_mode='categorical'
)

val_generator = val_datagen.flow_from_directory(
    val_dir,
    target_size=(299, 299),
    batch_size=32,
    class_mode='categorical'
)

Found 45596 images belonging to 39 classes.
Found 11379 images belonging to 39 classes.


In [8]:
import tensorflow as tf
from tensorflow.keras import layers, models

num_classes = 39 
input_shape = (299, 299, 3) 

base_eff = tf.keras.applications.EfficientNetV2L(
    input_shape=input_shape,
    include_top=False, 
    weights='imagenet'
)
base_eff.trainable = False 

model_eff = models.Sequential([
    base_eff,
    layers.GlobalAveragePooling2D(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(num_classes, activation='softmax', name="eff_output")
])

base_inc = tf.keras.applications.InceptionV3(
    input_shape=input_shape,
    include_top=False,
    weights='imagenet'
)
base_inc.trainable = False

model_inc = models.Sequential([
    base_inc,
    layers.GlobalAveragePooling2D(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(num_classes, activation='softmax', name="inc_output")
])

model_eff.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model_inc.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

print("✅ Both EfficientNetV2L and InceptionV3 are initialized.")

✅ Both EfficientNetV2L and InceptionV3 are initialized.


In [ ]:
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

checkpoint_eff = ModelCheckpoint('agrixai_eff_best.h5', save_best_only=True, monitor='val_accuracy')
checkpoint_inc = ModelCheckpoint('agrixai_inc_best.h5', save_best_only=True, monitor='val_accuracy')
early_stop = EarlyStopping(patience=5, restore_best_weights=True)

print("🚀 Training EfficientNetV2L...")
history_eff = model_eff.fit(
    train_generator,
    validation_data=val_generator,
    epochs=20,
    callbacks=[checkpoint_eff, early_stop]
)

print("🚀 Training InceptionV3...")
history_inc = model_inc.fit(
    train_generator,
    validation_data=val_generator,
    epochs=20,
    callbacks=[checkpoint_inc, early_stop]
)

🚀 Training EfficientNetV2L...
Epoch 1/20
   5/1425 ━━━━━━━━━━━━━━━━━━━━ 5:03:24 13s/step - accuracy: 0.0430 - loss: 3.6201